# Code Red Entity Resolution Pipeline

Run the cells from top to bottom. Each section is editable and corresponds to a pipeline phase. The index-building and inference cells are the expensive steps.


In [3]:
!git clone https://github.com/RealLakshay/business-record-linking.git
%cd business-record-linking/student_resource/code_red_submission/code/business_entity_resolution/src

Cloning into 'business-record-linking'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 40 (delta 5), reused 28 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 32.88 KiB | 4.11 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/business-record-linking/student_resource/code_red_submission/code/business_entity_resolution/src/business-record-linking/student_resource/code_red_submission/code/business_entity_resolution/src


In [4]:
%pip install rapidfuzz lightgbm scikit-learn

In [5]:
from pathlib import Path
import csv
import hashlib
import re
import sqlite3
from collections import Counter
from typing import Dict, Iterator, List, Sequence, Set, Tuple

from rapidfuzz import fuzz
try:
    import lightgbm as lgb
except ImportError:
    lgb = None
try:
    from sklearn.ensemble import HistGradientBoostingClassifier
except ImportError:
    HistGradientBoostingClassifier = None


def find_data_root():
    candidates = [Path.cwd(), Path('/content')]
    candidates.extend(path for base in list(candidates) if base.exists() for path in base.glob('*'))
    for base in candidates:
        if (base / 'dataset' / 'train' / 'train_source1.tsv').exists():
            return base
        nested = base / 'student_resource'
        if (nested / 'dataset' / 'train' / 'train_source1.tsv').exists():
            return nested
    raise FileNotFoundError('Could not find student_resource/dataset. Upload the student_resource folder or set DATA_ROOT manually.')


DATA_ROOT = find_data_root()
PROJECT_ROOT = DATA_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'code_red_submission' / 'code' / 'business_entity_resolution' / 'src'
if not SRC_DIR.exists():
    SRC_DIR = Path.cwd()
TRAIN_DIR = DATA_ROOT / 'dataset' / 'train'
TEST_DIR = DATA_ROOT / 'dataset' / 'test'
WORK_DIR = PROJECT_ROOT / 'code_red_submission' / 'work_notebook'
OUTPUT_DIR = PROJECT_ROOT / 'code_red_submission' / 'output'
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('data root:', DATA_ROOT.resolve())
print('project root:', PROJECT_ROOT.resolve())

data root: /content/business-record-linking/student_resource
project root: /content/business-record-linking


## Phase 0: audit and file reading


In [6]:
def read_rows(path: Path) -> Iterator[Dict[str, str]]:
    with path.open('r', encoding='utf-8', newline='') as handle:
        yield from csv.DictReader(handle, delimiter='\t')

def audit_file(path: Path):
    missing, countries = Counter(), Counter()
    rows = 0
    for row in read_rows(path):
        rows += 1
        for key, value in row.items():
            missing[key] += not (value or '').strip()
        if 'country' in row:
            countries[(row['country'] or '').strip()] += 1
    return rows, dict(missing), dict(countries)
for path in sorted(DATA_ROOT.glob('dataset/*/*.tsv')):
    print(path.name, audit_file(path))


test_source1.tsv (2, {'version https://git-lfs.github.com/spec/v1': 0}, {})
test_source2.tsv (2, {'version https://git-lfs.github.com/spec/v1': 0}, {})
test_source3.tsv (2, {'version https://git-lfs.github.com/spec/v1': 0}, {})
train_ground_truth.tsv (2, {'version https://git-lfs.github.com/spec/v1': 0}, {})
train_source1.tsv (2, {'version https://git-lfs.github.com/spec/v1': 0}, {})
train_source2.tsv (2, {'version https://git-lfs.github.com/spec/v1': 0}, {})
train_source3.tsv (2, {'version https://git-lfs.github.com/spec/v1': 0}, {})


## Phase 1: normalization


In [7]:
LEGAL = {'corp': 'corporation', 'co': 'company', 'inc': 'incorporated', 'ltd': 'limited', 'pvt': 'private', 'llc': 'limited liability company'}
STREET = {'rd': 'road', 'st': 'street', 'ave': 'avenue', 'blvd': 'boulevard', 'dr': 'drive'}


def normalize(value: str) -> str:
    value = (value or '').lower().replace('&', ' and ')
    value = re.sub(r'[^\w\s]', ' ', value, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', value).strip()


def normalize_name(value: str) -> str:
    return ' '.join(LEGAL.get(token, token) for token in normalize(value).split())


def normalize_address(value: str) -> str:
    return ' '.join(STREET.get(token, token) for token in normalize(value).split())


def row_features(row: Dict[str, str]) -> Dict[str, str]:
    name = normalize_name(row.get('business_name', ''))
    address = normalize_address(row.get('business_address', ''))
    return {
        'name': name,
        'name_sorted': ' '.join(sorted(name.split())),
        'address': address,
        'country': normalize(row.get('country', '')),
        'numbers': ' '.join(re.findall(r'\d+', address)),
        'name_prefix': name.split(' ', 1)[0][:4] if name else '',
    }


print(row_features({'business_name': 'Acme Corp & Sons', 'business_address': '10 Main Rd', 'country': 'US'}))

{'name': 'acme corporation and sons', 'name_sorted': 'acme and corporation sons', 'address': '10 main road', 'country': 'us', 'numbers': '10', 'name_prefix': 'acme'}


## Phase 2: blocking index and candidates


In [8]:
def create_index(db_path: Path, source_paths: Sequence[Path], reuse_existing: bool = True) -> None:
    print(f'[Phase 2] Starting index preparation: {db_path}')
    required_columns = {'id', 'name', 'name_sorted', 'address', 'country', 'numbers', 'name_prefix'}
    if db_path.exists() and reuse_existing:
        check_connection = None
        try:
            check_connection = sqlite3.connect(str(db_path), timeout=30)
            table_info = check_connection.execute('PRAGMA table_info(records)').fetchall()
            columns = {row[1] for row in table_info}
            integrity = check_connection.execute('PRAGMA integrity_check').fetchone()[0]
            if required_columns.issubset(columns) and integrity == 'ok':
                print('[Phase 2] Existing records table is valid; reusing index.')
                check_connection.close()
                return
            print(f'[Phase 2] Index needs rebuilding. integrity={integrity}; columns={sorted(columns)}')
        except sqlite3.DatabaseError as exc:
            print(f'[Phase 2] Existing database is malformed: {exc}; rebuilding it.')
        finally:
            if check_connection is not None:
                check_connection.close()
        old_connection = globals().get('train_connection')
        if old_connection is not None:
            try:
                old_connection.close()
                print('[Phase 2] Closed previous training connection.')
            except sqlite3.Error:
                pass
        db_path.unlink(missing_ok=True)
        print('[Phase 2] Removed old index.')

    connection = sqlite3.connect(str(db_path), timeout=30)
    connection.execute('PRAGMA journal_mode=OFF')
    connection.execute('PRAGMA synchronous=OFF')
    connection.execute('CREATE TABLE records (id TEXT PRIMARY KEY, name TEXT, name_sorted TEXT, address TEXT, country TEXT, numbers TEXT, name_prefix TEXT)')
    connection.execute('CREATE INDEX records_name ON records(name, country)')
    connection.execute('CREATE INDEX records_name_sorted ON records(name_sorted, country)')
    connection.execute('CREATE INDEX records_prefix ON records(name_prefix, country)')
    connection.execute('CREATE INDEX records_numbers ON records(numbers, country)')

    total_rows = 0
    batch = []
    for path in source_paths:
        print(f'[Phase 2] Reading source file: {path.name}')
        source_rows = 0
        for row in read_rows(path):
            values = row_features(row)
            batch.append((row['entity_id'], values['name'], values['name_sorted'], values['address'], values['country'], values['numbers'], values['name_prefix']))
            total_rows += 1
            source_rows += 1
            if len(batch) >= 10000:
                connection.executemany('INSERT INTO records VALUES (?, ?, ?, ?, ?, ?, ?)', batch)
                connection.commit()
                batch.clear()
            if source_rows % 100000 == 0:
                print(f'[Phase 2] {path.name}: {source_rows:,} rows read; total indexed: {total_rows:,}')
        print(f'[Phase 2] Finished {path.name}: {source_rows:,} rows')
    if batch:
        connection.executemany('INSERT INTO records VALUES (?, ?, ?, ?, ?, ?, ?)', batch)
    connection.commit()
    connection.close()
    print(f'[Phase 2] Index preparation complete: {total_rows:,} records indexed.')


def candidate_rows(connection, source_row: Dict[str, str], cap: int = 60):
    values = row_features(source_row)
    found = {}
    queries = [
        ("SELECT id,name,name_sorted,address,country,numbers FROM records WHERE name=? AND country=? LIMIT ?", (values['name'], values['country'], cap)),
        ("SELECT id,name,name_sorted,address,country,numbers FROM records WHERE name_sorted=? AND country=? LIMIT ?", (values['name_sorted'], values['country'], cap)),
        ("SELECT id,name,name_sorted,address,country,numbers FROM records WHERE name_prefix=? AND country=? LIMIT ?", (values['name_prefix'], values['country'], cap)),
    ]
    if values['numbers']:
        queries.append(("SELECT id,name,name_sorted,address,country,numbers FROM records WHERE numbers=? AND country=? LIMIT ?", (values['numbers'], values['country'], cap)))
    for sql, params in queries:
        for candidate in connection.execute(sql, params):
            found.setdefault(candidate[0], candidate)

    def rank(candidate):
        _, name, name_sorted, address, country, numbers = candidate
        return (
            int(name == values['name']) * 1000
            + int(name_sorted == values['name_sorted']) * 500
            + int(numbers and numbers == values['numbers']) * 300
            + fuzz.token_sort_ratio(values['name'], name)
            + fuzz.ratio(values['address'], address) * 0.25
        )

    return sorted(found.values(), key=rank, reverse=True)[:cap]


TRAIN_DB = WORK_DIR / 'train_source23.sqlite'
print('[Phase 2] Training S2/S3 index step')
create_index(TRAIN_DB, [TRAIN_DIR / 'train_source2.tsv', TRAIN_DIR / 'train_source3.tsv'])
train_connection = sqlite3.connect(TRAIN_DB, timeout=30)
print('[Phase 2] Training index connection ready:', TRAIN_DB)

[Phase 2] Training S2/S3 index step
[Phase 2] Starting index preparation: /content/business-record-linking/code_red_submission/work_notebook/train_source23.sqlite
[Phase 2] Reading source file: train_source2.tsv


KeyError: 'entity_id'

## Phase 3: feature engineering


In [ ]:
def similarity_features(left: Dict[str, str], right: Tuple[str, str, str, str, str, str]) -> List[float]:
    _, right_name, right_name_sorted, right_address, right_country, right_numbers = right
    left_values = row_features(left)
    left_name, left_address = left_values['name'], left_values['address']
    left_tokens, right_tokens = set(left_name.split()), set(right_name.split())
    address_tokens, other_address_tokens = set(left_address.split()), set(right_address.split())
    return [
        float(fuzz.ratio(left_name, right_name)) / 100,
        float(fuzz.token_sort_ratio(left_name, right_name)) / 100,
        float(left_values['name_sorted'] == right_name_sorted),
        len(left_tokens & right_tokens) / (len(left_tokens | right_tokens) or 1),
        float(fuzz.ratio(left_address, right_address)) / 100,
        len(address_tokens & other_address_tokens) / (len(address_tokens | other_address_tokens) or 1),
        float(bool(left_values['numbers'] and left_values['numbers'] == right_numbers)),
        float(left_values['country'] == right_country),
        float(not left_address or not right_address),
        float(abs(len(left_tokens) - len(right_tokens))),
    ]

with (TRAIN_DIR / 'train_source1.tsv').open(encoding='utf-8', newline='') as handle:
    first_s1 = next(csv.DictReader(handle, delimiter='\t'))
example_candidates = candidate_rows(train_connection, first_s1, cap=60)
print('candidate count:', len(example_candidates))
if example_candidates:
    print('features:', similarity_features(first_s1, example_candidates[0]))

## Phase 4: labels and training sample


In [ ]:
def truth_map(path: Path):
    return {row['source1_entity_id']: {item for item in row['matched_entity_ids'].split(',') if item} for row in read_rows(path)}


def make_training_data(s1_path, gt_path, connection, cap=60, sample_size=100000):
    labels = truth_map(gt_path)
    features, targets, groups = [], [], []
    for row in read_rows(s1_path):
        digest = int(hashlib.md5(row['entity_id'].encode()).hexdigest(), 16)
        if digest % 10 == 7 or digest % 100000 >= max(1, sample_size // 25):
            continue
        actual = labels.get(row['entity_id'], set())
        for candidate in candidate_rows(connection, row, cap):
            features.append(similarity_features(row, candidate))
            targets.append(int(candidate[0] in actual))
            groups.append(row['entity_id'])
    if not features:
        raise RuntimeError('No training candidates were generated')
    return features, targets, groups


CANDIDATE_CAP = 60
features, targets, groups = make_training_data(TRAIN_DIR / 'train_source1.tsv', TRAIN_DIR / 'train_ground_truth.tsv', train_connection, CANDIDATE_CAP, 100000)
print('training pairs:', len(targets), 'positive pairs:', sum(targets))

## Phase 5: train the matching classifier


In [ ]:
if lgb is not None:
    model = lgb.LGBMClassifier(n_estimators=350, learning_rate=0.05, num_leaves=31, max_depth=8, random_state=17, verbosity=-1)
elif HistGradientBoostingClassifier is not None:
    model = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.08, max_leaf_nodes=31, random_state=17)
else:
    raise RuntimeError('Install lightgbm or scikit-learn')
model.fit(features, targets)
print('trained:', type(model).__name__)

def predict_probability(model, feature_rows):
    return [float(item[1]) for item in model.predict_proba(feature_rows)] if hasattr(model, 'predict_proba') else [float(item) for item in model.predict(feature_rows)]


## Phase 6: tune threshold using macro F0.5


In [ ]:
def f05(actual: Set[str], predicted: Set[str]) -> float:
    if not actual and not predicted:
        return 1.0
    if not actual or not predicted:
        return 0.0
    true_positives = len(actual & predicted)
    precision = true_positives / len(predicted)
    recall = true_positives / len(actual)
    denominator = 0.25 * precision + recall
    if denominator == 0:
        return 0.0
    return 1.25 * precision * recall / denominator


def tune_threshold(model, s1_path, gt_path, connection, cap=30, validation_mod=7):
    print('[Phase 6] Loading validation labels')
    labels = truth_map(gt_path)
    validation = []
    validation_rows = 0
    for row in read_rows(s1_path):
        if int(hashlib.md5(row['entity_id'].encode()).hexdigest(), 16) % 10 != validation_mod:
            continue
        candidates = candidate_rows(connection, row, cap)
        probabilities = predict_probability(model, [similarity_features(row, c) for c in candidates])
        validation.append((labels.get(row['entity_id'], set()), [(c[0], p) for c, p in zip(candidates, probabilities)]))
        validation_rows += 1
        if validation_rows % 100000 == 0:
            print(f'[Phase 6] Validation rows scored: {validation_rows:,}')
    print(f'[Phase 6] Validation rows complete: {validation_rows:,}')

    best_threshold, best_score = 0.5, -1
    for threshold in [x / 100 for x in range(30, 100, 2)]:
        score = sum(
            f05(actual, {item for item, probability in scored if probability >= threshold})
            for actual, scored in validation
        ) / max(1, len(validation))
        if score > best_score:
            best_threshold, best_score = threshold, score
    print(f'[Phase 6] Best threshold: {best_threshold:.2f}; macro F0.5: {best_score:.6f}')
    return best_threshold, best_score


threshold, validation_score = tune_threshold(
    model,
    TRAIN_DIR / 'train_source1.tsv',
    TRAIN_DIR / 'train_ground_truth.tsv',
    train_connection,
    CANDIDATE_CAP,
)
print('validation macro F0.5:', validation_score)
print('selected threshold:', threshold)

## Test inference and output generation

Run this last. It creates both required submission TSV files.


In [ ]:
def write_outputs(model, threshold, s1_path, connection, cap, output_dir):
    print('[Inference] Starting test output generation')
    output_dir.mkdir(parents=True, exist_ok=True)
    matching_path = output_dir / 'matching_results.tsv'
    candidate_path = output_dir / 'candidate_pairs.tsv'
    processed = 0
    empty_candidates = 0
    with matching_path.open('w', encoding='utf-8', newline='') as matching, candidate_path.open('w', encoding='utf-8', newline='') as candidates:
        matching_writer = csv.writer(matching, delimiter='\t', lineterminator='\n')
        candidate_writer = csv.writer(candidates, delimiter='\t', lineterminator='\n')
        matching_writer.writerow(['source1_entity_id', 'matched_entity_ids'])
        candidate_writer.writerow(['source1_entity_id', 'candidate_entity_ids'])
        for row in read_rows(s1_path):
            possible = candidate_rows(connection, row, cap)
            ids = [candidate[0] for candidate in possible]
            if possible:
                feature_rows = [similarity_features(row, candidate) for candidate in possible]
                probabilities = predict_probability(model, feature_rows)
                matches = [item for item, probability in zip(ids, probabilities) if probability >= threshold]
            else:
                probabilities = []
                matches = []
                empty_candidates += 1
            candidate_writer.writerow([row['entity_id'], ','.join(ids)])
            matching_writer.writerow([row['entity_id'], ','.join(matches)])
            processed += 1
            if processed % 100000 == 0:
                print(f'[Inference] S1 rows processed: {processed:,}; empty candidate sets: {empty_candidates:,}')
    print(f'[Inference] Complete: {processed:,} S1 rows; empty candidate sets: {empty_candidates:,}')
    print(f'[Inference] Matching file: {matching_path}')
    print(f'[Inference] Candidate file: {candidate_path}')


TEST_DB = WORK_DIR / 'test_source23.sqlite'
old_test_connection = globals().get('test_connection')
if old_test_connection is not None:
    try:
        old_test_connection.close()
        print('[Inference] Closed previous test connection.')
    except sqlite3.Error:
        pass
print('[Inference] Preparing test S2/S3 index')
create_index(TEST_DB, [TEST_DIR / 'test_source2.tsv', TEST_DIR / 'test_source3.tsv'])
test_connection = sqlite3.connect(TEST_DB, timeout=30)
write_outputs(model, threshold, TEST_DIR / 'test_source1.tsv', test_connection, CANDIDATE_CAP, OUTPUT_DIR)
test_connection.close()
print('[Inference] Test connection closed.')